# TreeXplain - Data Exploration Notebook

This notebook demonstrates how to explore satellite data and visualize deforestation patterns.

In [ ]:
# Setup
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)

## 1. Import TreeXplain Modules

In [ ]:
from treexplain.config import DATA_DIR, MODELS_DIR, OUTPUT_DIR, INPUT_BANDS, IMAGE_SIZE
from treexplain.data.preprocessor import SatellitePreprocessor
from treexplain.data.pipeline import DataPipeline, DeforestationDataset
from treexplain.model.deforestation import DeforestationModel
from treexplain.xai.explainer import TreeXplainExplainer

print('TreeXplain modules imported successfully!')

## 2. Create Sample Data for Exploration

In [ ]:
# Create sample satellite-like images
def create_sample_image(size=256, num_channels=6):
    '''Create a sample satellite image with realistic patterns.'''
    image = np.zeros((num_channels, size, size), dtype=np.float32)
    
    # Create coordinate grids
    x, y = np.meshgrid(np.linspace(0, 1, size), np.linspace(0, 1, size))
    
    # Different patterns for different bands
    for c in range(num_channels):
        # Base pattern
        base = 0.3 + 0.4 * (x * y)
        
        # Add some noise
        noise = 0.1 * np.random.randn(size, size)
        
        # Add circular features (like forests)
        center_x, center_y = 0.5, 0.5
        radius = 0.3
        circle = ((x - center_x)**2 + (y - center_y)**2) < radius**2
        
        # Combine
        image[c] = base + noise + 0.2 * circle
        
        # Normalize to [0, 1]
        image[c] = np.clip(image[c], 0, 1)
    
    return image

# Create sample images
num_samples = 12
images = [create_sample_image() for _ in range(num_samples)]
labels = [1 if i % 3 == 0 else 0 for i in range(num_samples)]  # 1 = deforestation

print(f'Created {len(images)} sample images')
print(f'Image shape: {images[0].shape}')
print(f'Labels: {labels}')

## 3. Visualize Sample Images

In [ ]:
def plot_images(images, labels, num_rows=3, num_cols=4):
    '''Plot a grid of images with labels.'''
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 10))
    axes = axes.ravel()
    
    for i in range(min(len(images), num_rows * num_cols)):
        ax = axes[i]
        
        # Select a few bands to display (RGB-like)
        if images[i].shape[0] >= 3:
            rgb = images[i][2:5]  # Use bands 3, 4, 5 (approximately RGB)
            rgb = np.transpose(rgb, (1, 2, 0))  # CHW to HWC
            ax.imshow(rgb)
        else:
            ax.imshow(images[i][0], cmap='gray')
        
        label = 'Deforestation' if labels[i] == 1 else 'Forest'
        ax.set_title(f'Label: {label}')
        ax.axis('off')
    
    # Hide unused subplots
    for i in range(len(images), num_rows * num_cols):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot sample images
plot_images(images, labels)

## 4. Test the Model

In [ ]:
# Create model
model = DeforestationModel()
print(f'Model has {sum(p.numel() for p in model.parameters()):,} parameters')

# Test with sample input
sample_input = torch.from_numpy(images[0]).float().unsqueeze(0)
print(f'Input shape: {sample_input.shape}')

# Make prediction
with torch.no_grad():
    output = model(sample_input)
    probability = output.item()
    prediction = 1 if probability > 0.5 else 0

print(f'Prediction: {prediction} (Deforestation' if prediction == 1 else 'Forest)')
print(f'Probability: {probability:.4f}')

## 5. Generate Explanations

In [ ]:
# Create explainer
explainer = TreeXplainExplainer(model)

# Set background for SHAP (using zeros for demo)
background = torch.zeros(5, 6, 256, 256)
explainer.set_background(background)

# Generate GradCAM explanation
gradcam = explainer.explain_gradcam(sample_input)
print(f'GradCAM shape: {gradcam.shape}')

# Visualize GradCAM
def plot_gradcam(gradcam, original_image, alpha=0.5):
    '''Plot GradCAM overlay on original image.'''
    # Take mean across channels
    gradcam_2d = gradcam.mean(dim=1).squeeze().numpy()
    gradcam_2d = np.clip(gradcam_2d, 0, 1)
    
    # Resize to match original image
    from skimage.transform import resize
    if gradcam_2d.shape != original_image.shape[1:]:
        gradcam_2d = resize(gradcam_2d, original_image.shape[1:])
    
    # Create RGB version of original
    if original_image.shape[0] >= 3:
        rgb = original_image[2:5]  # Use bands 3, 4, 5
        rgb = np.transpose(rgb, (1, 2, 0))
    else:
        rgb = np.stack([original_image[0]] * 3, axis=-1)
    
    # Normalize RGB for display
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    
    # Create heatmap
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
    
    ax1.imshow(rgb)
    ax1.set_title('Original Image')
    ax1.axis('off')
    
    ax2.imshow(gradcam_2d, cmap='jet')
    ax2.set_title('GradCAM')
    ax2.axis('off')
    
    # Overlay
    overlay = rgb.copy()
    heatmap = plt.cm.jet(gradcam_2d)[:, :, :3]
    overlay = 0.7 * rgb + 0.3 * heatmap
    overlay = np.clip(overlay, 0, 1)
    
    ax3.imshow(overlay)
    ax3.set_title('GradCAM Overlay')
    ax3.axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot GradCAM
plot_gradcam(gradcam, images[0])

## 6. Create Dataset and DataLoader

In [ ]:
# Create dataset
dataset = DeforestationDataset(images, labels, augment=True)
print(f'Dataset size: {len(dataset)}')

# Create DataLoader
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Test a batch
batch = next(iter(dataloader))
batch_images, batch_labels = batch
print(f'Batch images shape: {batch_images.shape}')
print(f'Batch labels: {batch_labels}')

# Make predictions on batch
with torch.no_grad():
    outputs = model(batch_images)
    predictions = (outputs > 0.5).float().squeeze()
    print(f'Batch predictions: {predictions}')

## 7. Save and Load Model

In [ ]:
# Save model
model_path = OUTPUT_DIR / 'demo_model.pt'
torch.save(model.state_dict(), model_path)
print(f'Model saved to {model_path}')

# Load model
new_model = DeforestationModel()
new_model.load_state_dict(torch.load(model_path))
print('Model loaded successfully!')

# Test loaded model
with torch.no_grad():
    output = new_model(sample_input)
    probability = output.item()
    prediction = 1 if probability > 0.5 else 0
    print(f'Loaded model prediction: {prediction}, probability: {probability:.4f}')

## 8. Next Steps

- Connect to real STAC API for satellite data
- Train the model on real deforestation data
- Deploy the API to a cloud service
- Create more sophisticated XAI visualizations
- Build a complete web interface